# Curry-Howard 对应

```{note}
*lagniappe* 是一份小而意想不到的礼物 &mdash; 一点“额外的东西”。
请享受这一小章，其中包含最美丽的内容之一
整本书中最优美的结果之一。本章基于论文
[Propositions as Types][pat] 作者：菲利普·瓦德勒。你可以观看有趣的节目
另外，Wadler 教授的 [recorded lecture][pat-lambda-days]
下面我们的讲座。
```

{{ video_embed | replace("%%VID%%", "GdcOy6zVFC4")}}

正如我们很久以前观察到的，OCaml 是 ML 家族中的一种语言，而 ML 是
最初设计为定理的 <u>m</u>eta <u>l</u>anguage
证明器&mdash;即一个旨在帮助证明和检查
逻辑公式的证明。在构造证明时，最好是
确保你只能证明真实的公式，以确保你不会做出
不正确的论据等

我们的梦想是拥有一个可以判定任意逻辑公式真假的计算机程序。
对于某些公式，这是可能的。但是，20 世纪初的突破性结果之一是：
一般来说，计算机程序*无法*做到这一点。Alonzo Church 和 Alan
图灵在 1936 年独立地证明了这一点。丘奇使用 *lambda 演算* 作为
计算机模型；图灵使用了我们现在所说的“图灵机”。
*丘奇-图灵论题*是一个假设，它认为 lambda 演算和图灵机
都形式化了“计算”这一非正式概念。

我们不会专注于那个不可能的任务，而是专注于
证明与程序之间的关系。事实证明，二者以一种令人惊讶的方式深刻相连。

## 用证据计算

我们习惯使用 OCaml 程序来操作数据，例如整数和
变体和函数。这些数据值总是被键入：在编译时，
OCaml 推断（或程序员注释）表达式的类型。对于
例如，`3110 : int` 和 `[] : 'a list`。我们很久以前就学会了把它们读作
“`3110` 具有类型 `int`”，“`[]` 具有类型 `'a list`”。

现在让我们尝试不同的读法。不要读作“具有类型”，而读作“是……的证据”。
因此，`3110` 是 `int` 的证据。这意味着什么？可以把类型想成一组值。
因此，`3110` 是类型 `int` 非空的证据。
同样，`[]` 是类型 `'a list` 不为空的证据。我们说
如果类型不为空，我们说这个类型是*有居住元的*。

有空类型吗？ OCaml 中实际上有一个，尽管我们从未有过
之前提到的理由。可以定义一个没有的变体类型
构造函数：

In [ ]:
type empty = |

我们可以将该类型称为任何我们想要的名称，而不是 `empty`；的
特殊语法只是编写 `|` 而不是实际的构造函数。 （注意，
该语法可能会给一些编辑带来一些麻烦。你可能需要把
后面加双分号以确保格式正确。）这是不可能的
构造一个 `empty` 类型的值，正是因为它没有构造函数。所以，
`empty` 没有居住元。

根据我们基于证据的新解读，我们可以将函数视为方法
操纵和转换证据&mdash;就像我们已经习惯的那样
将函数视为操纵和转换数据的方法。例如，
以下函数构造和析构对：

In [ ]:
let pair x y = (x, y)
let fst (x, y) = x
let snd (x, y) = y

我们可以将 `pair` 视为一个接受 `'a` 证据的函数，并且
`'b` 的证据，并返回 `'a * 'b` 的证据。后一份证据
是包含各个证据的 `(x, y)` 对，
`x` 和 `y`。类似地， `fst` 和 `snd` 提取各个部分
二者的证据。因此，

- 如果你有 `'a` 的证据和 `'b` 的证据，你可以出示证据
对于 `'a` 和 `'b`。
- 如果你有 `'a` 和 `'b` 的证据，那么你可以提供以下证据：
  `'a`.
- 如果你有 `'a` 和 `'b` 的证据，那么你可以提供以下证据：
  `'b`.

在学习证明时（例如，在离散数学课上），你已经学到：
为了证明两个陈述都成立，需要分别证明每个陈述成立。
也就是说，要证明 A 和 B 的合取，
你必须证明 A 并证明 B。同样，如果你有
A 和 B 的合取，则可以得出 A 成立，并且可以得出 B 成立
成立。  我们可以将这些推理模式写成逻辑公式，使用
`/\` 表示合取，`->` 表示蕴涵：

```text
A -> B -> A /\ B
A /\ B -> A
A /\ B -> B
```

证明是证据的一种形式：它们是关于真理的逻辑论证
的一份声明。因此，这些公式的另一种解读是：

- 如果你有A的证据和B的证据，你就可以出示证据
对于 A 和 B。
- 如果你有 A 和 B 的证据，那么你就可以提供 A 的证据。
- 如果你有A和B的证据，那么你就可以提供B的证据。

请注意，我们现在给程序和证明赋予了相同的读法。
它们都是操纵和转换证据的方式。事实上，仔细
比较 `pair`、`fst` 和 `snd` 的类型与逻辑公式
描述有效的推理模式：

```text
val pair : 'a -> 'b -> 'a * 'b         A -> B -> A /\ B
val fst : 'a * 'b -> 'a                A /\ B -> A
val snd : 'a * 'b -> 'b                A /\ B -> B
```

如果将 `'a` 替换为 A，将 `'b` 替换为 B，将 `*` 替换为 `/\`，则 **
程序与公式相同！**

## 对应关系

我们刚刚发现的是，带有证据的计算对应于
构建有效的逻辑证明。这种对应并不只是这三个特定程序的偶然现象。
相反，这是一个深刻现象，
连接编程和逻辑领域。它的各个方面已经
被许多领域的许多人发现。因此，它有很多名字。一
常用名称是*库里-霍华德对应关系*，以逻辑学家 Haskell
Curry（函数式编程语言 Haskell 就是以他的名字命名的）和
William Howard 命名。这个对应关系将编程中的思想与
逻辑：

- 类型对应于逻辑公式（又名*命题*）。
- 程序对应于逻辑证明。
- 求值对应于证明的化简。

我们已经看过其中的前两个对应。我们的类型
三个小程序对应于公式，以及程序本身
对应于涉及连词的证明中进行的推理。我们还没有
还没看到第三个；我们稍后会。

让我们深入研究每个对应，以更全面地欣赏它们。

## 类型对应于命题

在*命题逻辑*中，公式是用原子命题创建的，
否定、合取、析取和蕴含。下面的BNF描述了
命题逻辑公式：

```text
p ::= atom
    | ~ p      (* negation *)
    | p /\ p   (* conjunction *)
    | p \/ p   (* disjunction *)
    | p -> p   (* implication *)

atom ::= <identifiers>
```

例如，`raining /\ snowing /\ cold` 是一个命题，表明它是
同时下雨、下雪和寒冷（这种天气条件称为
[*Ithacating*][ithacating])。一个原子命题可能适用于世界，或者
不。有两个不同的原子命题，写为 `true` 和
`false`，分别是始终保持和从不保持。

[ithacating]: https://www.urbandictionary.com/define.php?term=Ithacating

所有这些*连接词*（之所以这样称呼，是因为它们将公式连接在一起）
函数式程序类型的对应关系。

**连词。** 我们已经看到 `/\` 连接词对应于
`*` 类型构造函数。命题 `A /\ B` 断言 `A` 和
`B`。 `a * b` 类型的 OCaml 值包含 `a` 和 `b` 类型的值。
因此，`/\` 和 `*` 都对应于配对或产品的概念。

**蕴涵。** 蕴含连接词 `->` 对应于函数
类型构造函数 `->`。命题 `A -> B` 断言如果你能证明
`A` 成立，那么你可以证明 `B` 成立。换句话说，假设 `A`，
你可以得出 `B` 的结论。从某种意义上说，这意味着你可以将 `A` 转换为 `B`。
`a -> b` 类型的 OCaml 值更清楚地表达了这个想法。这样的值
是一个将 `a` 类型的值转换为 `b` 类型的值的函数。
因此，如果你可以证明 `a` 有居住元（通过展示该类型的值），
你也可以证明 `b` 有居住元（通过把 `a -> b` 类型的函数应用到它）。
所以，`->` 对应的是变换的思想。

**析取。** 析取连接词 `\/` 对应的概念
在 OCaml 中较难简洁表达。命题 `A \/ B`
断言你可以证明 `A` 成立或 `B` 成立。让我们把它进一步加强为：
除了证明其中*一个*成立之外，你还必须
指定你要显示的*哪一个*。为什么这很重要？

假设我们正在研究“孪生素数猜想”的证明，这是一个尚未解决的问题
问题指出存在无限多个孪生素数（素数形式为
$n$ 和 $n+2$，例如 3 和 5，或 5 和 7)。让原子命题`TP`
表示有无穷多个孪生素数。那么命题
`TP \/ ~ TP` 似乎是合理的：要么有无限多个孪生素数，要么
没有。我们甚至不需要弄清楚如何证明这个猜想！
但如果我们将 `\/` 的含义加强为我们必须说明*哪个*
两边（左或右）成立，那么我们要么必须给出证明，要么
反驳猜想。目前没有人知道如何做到这一点。所以我们可以
不能证明 `TP \/ ~ TP`。

从今以后，我们将使用 `\/`，并要求必须确定
我们正在给出左侧或右侧命题的证明。因此，我们不能
对于任何命题 `p` 必然得出 `p \/ ~ p` 的结论：是否重要
我们能否自己证明 `p` 或 `~ p`。从技术上讲，这使得我们的
命题逻辑是*构造性的*而不是*经典的*。在构造性
逻辑中，我们必须构造各个命题的证明。经典
逻辑（理解 `\/` 的传统方式）不需要这样做。

回到析取和变体之间的对应关系，考虑这个
变体类型：

In [ ]:
type ('a, 'b) disj = Left of 'a | Right of 'b

该类型的值 `v` 可以是 `Left a`，其中 `a : 'a`；或 `Right b`，其中
`b : 'b`。也就是说，`v` 标明 (i) 它使用的是左构造函数
还是右构造函数，并且 (ii) 它恰好包含一个
类型为 `'a` 或 `'b`&mdash; 的子值，而不是这两种类型的两个子值，
这就是 `'a * 'b` 。

因此，（构造性）析取连接词 `\/` 对应于 `disj`
类型构造函数。命题 `A \/ B` 断言 `A` 或 `B` 成立
以及它是哪一个，左还是右。类型为 `('a, 'b) disj` 的 OCaml 值
类似地包含 `'a` 或 `'b` 类型的值以及标识
（使用 `Left` 或 `Right` 构造函数）它是哪一个。 `\/` 和 `disj`
因此对应于并集的思想。

**真与假。** 原子命题 `true` 是唯一的命题
保证始终保持。 OCaml 中有很多类型总是
有居住元，但其中最简单的是 `unit`：有一个值 `()` 属于
`unit`。因此命题 `true`（最自然地）对应于类型 `unit`。

同样，原子命题 `false` 是唯一的命题
保证永远不成立。它对应于我们之前引入的 `empty` 类型
早些时候，它没有构造函数。 （该类型的其他名称可能包括
`zero` 或 `void`，但我们将坚持使用 `empty`。）

我们应该解决 `empty` 的一个微妙之处。类型没有
构造函数，但仍然可以编写具有类型的表达式
`empty`。这是一种方法：

In [ ]:
let rec loop x = loop x

现在，如果你在 utop 中输入此代码，你将不会得到任何响应：

```ocaml
let e : empty = loop ()
```

该表达式类型检查成功，然后进入无限循环。所以，
永远不会产生任何 `empty` 类型的值，即使
表达式具有该类型。

这是另一种方法：

In [ ]:
let e : empty = failwith ""

同样，表达式类型检查，但它永远不会生成类型的实际值
`empty`。相反，这次产生了异常。

所以类型 `empty` 没有居住元，即使有一些表达式具有该类型。
但是，**如果我们要求程序是全的**，就可以排除那些表达式。
这意味着排除会引发异常或进入无限循环的程序。
事实上，当我们开始讨论形式化方法时，我们确实提出了这个要求，并将继续假设它。

**否定。** 这个连接词是最棘手的。让我们考虑否定
实际上是语法糖。特别地，我们假设命题
公式 `~ p` 实际上意味着这个公式：`p -> false`。为什么？
公式 `~ p` 应该意味着 `p` 不成立。因此，如果 `p` *确实*成立，那么它
会导致矛盾。因此，给定 `p`，我们可以得出 `false` 的结论。这个
是理解构造逻辑中否定的标准方式。

鉴于这种语法糖，`~ p` 因此对应于一个函数类型，其
返回类型为 `empty`。这样的函数永远不会真正返回。鉴于我们的
持续假设程序是全的，这必定意味着不可能
甚至调用该函数。因此，不可能构造出
函数的输入类型。因此否定对应于以下想法
不可能，或者矛盾。

**作为类型的命题。** 我们现在创建了以下对应关系
使我们能够将命题视为类型：

- `/\` 和 `*`
- `->` 和 `->`
- `\/` 和 `disj`
- `true` 和 `unit`
- `false` 和 `empty`
- `~` 和 `... -> false`

但这只是库里-霍华德对应关系的第一层。它还会更深入……

## 程序对应于证明

我们已经看到，程序和证明都是操纵和改造的方式
证据。事实上，每个程序**都是**该程序类型的证明
有居住元，因为类型检查器必须验证程序的类型是否正确。

然而，类型检查的细节导致了更引人注目的结果
程序和证明之间的对应关系。让我们将注意力限制在
仅涉及合取和蕴涵的程序和证明，或者同等地，
对和函数。（其他命题连接词也可以纳入进来，但需要额外工作。）

**类型检查规则。** 对于类型检查，我们给出了许多*规则*来定义何时
程序类型良好。以下是变量、函数和对的规则：

```text
{x : t, ...} |- x : t
```
变量具有环境为它指定的类型。

```text
env |- fun x -> e : t -> t'
if env[x -> t] |- e : t'
```

如果 `e` 在 a 中具有类型 `t'`，则匿名函数 `fun x -> e` 具有类型 `t -> t'`
静态环境扩展为将 `x` 绑定到类型 `t`。

```text
env |- e1 e2 : t'
if env |- e1 : t -> t'
and env |- e2 : t
```

如果 `e1` 具有类型 `t -> t'` 并且 `e2` 具有类型，则应用程序 `e1 e2` 具有类型 `t'`
类型 `t`。

```text
env |- (e1, e2) : t1 * t2
if env |- e1 : t1
and env |- e2 : t2
```

如果 `e1` 具有类型 `t1` 并且 `e2` 具有类型，则 `(e1, e2)` 对具有类型 `t1 * t2`
类型 `t2`。

```text
env |- fst e : t1
if env |- e : t1 * t2

env |- snd e : t2
if env |- e : t1 * t2
```

如果 `e` 具有类型 `t1 * t2`，则 `fst e` 具有类型 `t1`，并且 `snd e` 具有类型
`t2`。

**证明树。** 表达这些规则的另一种方式是绘制*证明
显示规则的递归应用的树*。这是那些证明树：

```text

---------------------
{x : t, ...} |- x : t


   env[x -> t1] |- e2 : t2
-----------------------------
env |- fun x -> e2 : t1 -> t2


env |- fun x -> e2 : t1 -> t2        env |- e1 : t1
---------------------------------------------------
          env |- (fun x -> e2) e1 : t2


env |- e1 : t1         env |- e2 : t2
-------------------------------------
     env |- (e1, e2) : t1 * t2


env |- e : t1 * t2
------------------
env |- fst e : t1


env |- e : t1 * t2
------------------
env |- snd e : t2
```

**从逻辑上看证明树。** 让我们重写每个证明树以消除
程序，只留下类型。同时，我们使用
命题作为类型对应将类型重写为命题：

```text

-----------
env, p |- p


 env, p1 |- p2
---------------
env |- p1 -> p2


env |- p1 -> p2     env |- p1
-----------------------------
        env |- p2


env |- p1      env |- p2
------------------------
    env |- p1 /\ p2


env |- p1 /\ p2
---------------
   env |- p1


env |- p1 /\ p2
---------------
   env |- p2
```

现在，每条规则都可以被解读为逻辑推理的有效形式。每当我们
写 `env |- t`，意味着“根据 `env` 中的假设，我们可以得出结论
`p` 成立”。像往常一样，规则意味着从该线上方的所有前提出发，
线下结论成立。

**证明和程序。** 现在考虑以下证明树，显示
程序类型的推导：

```text
------------------------           ------------------------
{p : a * b} |- p : a * b           {p : a * b} |- p : a * b
------------------------           ------------------------
{p : a * b} |- snd p : b           {p : a * b} |- fst p : a
-----------------------------------------------------------
        {p : a * b} |- (snd p, fst p) : b * a
     ----------------------------------------------
     {} |- fun p -> (snd p, fst p) : a * b -> b * a
```

该程序表明你可以交换一对组件，因此
交换涉及的类型。

如果我们删除程序，只留下类型，然后重写它们
作为命题，我们得到这个证明树：

```text
----------------           ----------------
a /\ b |- a /\ b           a /\ b |- a /\ b
----------------           ----------------
  a /\ b |- b                a /\ b |- a
-------------------------------------------
              a /\ b |- b /\ a
           ----------------------
           {} |- a /\ b -> b /\ a
```

这是命题逻辑的有效证明树。这表明你可以
交换连词的两边。

我们从这两个证明树中看到的是：**程序就是证明**。一个
类型良好的程序对应于逻辑命题的证明。它显示
如何用证据进行计算，在本例中将 `a /\ b` 的证明转换为
`b /\ a` 的证明。

**程序就是证明。** 我们现在创建了以下对应关系
使我们能够将程序作为证明来阅读：

- 程序 `e : t` 对应于类型 `t` 所对应的逻辑公式的证明。
- `|- t` 的证明树对应于 `{} |- e : t` 的证明树。
- 给程序定型的证明规则对应于证明命题的规则。

但这只是库里-霍华德对应关系的第二层。它还会更深入……

## 求值对应于化简

我们将仅简要处理这一部分对应。考虑
以下程序：

```ocaml
fst (a, b)
```

该程序当然会求值为 `a`。

接下来，考虑该程序的类型推导。  变量 `a` 和
`b` 必须绑定到程序静态环境中的某些类型
类型检查。

```text
------------------------         -------------------------
{a : t, b : t'} |- a : t         {a : t, b : t'} |- b : t'
----------------------------------------------------------
         {a : t, b : t'} |- (a, b) : t * t'
         ----------------------------------
         {a : t, b : t'} |- fst (a, b) : t
```

根据程序即证明的对应关系，删除证明树中的程序、只保留命题，
我们得到这个证明树：
```text
-----------            -----------
t, t' |-  t            t, t' |- t'
----------------------------------
         t, t' |- t /\ t'
         ----------------
            t, t' |- t
```

然而，有一个更简单的证明树，具有相同的结论：

```text
----------
t, t' |- t
```

换句话说，如果 `t` 已经是一个假设，我们不需要绕道先证明 `t /\ t'`
再证明 `t`。我们可以直接得出 `t`。

同样，有一个更简单的类型推导对应于相同的
更简单的证明：

```text
------------------------
{a : t, b : t'} |- a : t
```

请注意，这个类型推导针对的是程序 `a`，而它正是更大的程序
`fst (a, b)` 的求值结果。

因此，程序的求值会导致证明树化简，而化简后的证明树实际上是
（通过程序即证明的对应关系）同一命题的更简单证明。
**因此，求值对应于证明化简。** 这就是库里-霍华德对应关系的最终层次。

## 这一切意味着什么

逻辑是人类探究的一个基本方面。  它指导我们推理
世界，在得出有效的推论时，在推论什么是必须真实的与什么是真实的时
一定是假的。  各个领域的逻辑和论证训练&mdash;
学科&mdash;是高等教育最重要的部分之一。

柯里-霍华德对应关系表明逻辑和计算是
以一种深刻甚至神秘的方式从根本上联系在一起。基本建筑
逻辑构建块（命题、证明）对应于基本的
计算构建块（类型、函数式程序）。计算本身，
也就是表达式的求值或化简，对应于证明的化简。
因此，计算机执行的任务，与人类试图以尽可能好的方式给出证明的任务相通。

因此，计算与推理有着内在的联系。而函数式
编程是人类探究的基本组成部分。

是否有更好的理由来学习函数式编程？

[pat]: http://homepages.inf.ed.ac.uk/wadler/papers/propositions-as-types/propositions-as-types.pdf
[pat-lambda-days]: https://www.youtube.com/watch?v=aeRVdYN6fE8

## 练习

{{ solutions }}

<!--------------------------------------------------------------------------->
{{ ex2 | replace("%%NAME%%", "propositions as types")}}

对于以下每个命题，写出其相应的类型
在 OCaml 中。

- `true -> p`
- `p /\ (q /\ r)`
- `(p \/ q) \/ r`
- `false -> p`

<!--------------------------------------------------------------------------->
{{ ex3 | replace("%%NAME%%", "programs as proofs")}}

对于以下每个命题，确定其对应的类型
OCaml，然后编写一个 OCaml `let` 定义来给出该类型的程序。你的
程序证明该类型是 *inhabited*，也就是存在一个该类型的值。
这也证明了对应命题成立。

- `p /\ q -> q /\ p`
- `p \/ q -> q \/ p`

<!--------------------------------------------------------------------------->
{{ ex3 | replace("%%NAME%%", "evaluation as simplification")}}

考虑以下 OCaml 程序：

```ocaml
let f x = snd ((fun x -> x, x) (fst x))
```

- 该程序的类型是什么？
- 该类型对应的命题是什么？
- `f (1,2)` 在小步语义中将如何求值？
- 该求值建议对 `f` 进行怎样的简化实现？（或者
也许有几个，但其中一个可能是最简单的？）
- 你的简化后的 `f` 仍然具有与原始类型相同的类型吗？（它
应该。）

你的简化 `f` 和原始 `f` 都是相同的证明
命题，但求值可以帮助你产生更简单的证明。